# 🌙 10. MAGSAC++ Robust Geometric Outlier Rejection

**Mission Context**: Marginalizing Sample Consensus (MAGSAC++) for robust geometric estimation in the presence of extreme outlier noise and terrain repetitiveness.  
**Objectives**:
- Filter raw correspondence points using `cv2.USAC_MAGSAC`.
- Calculate Inlier Count, Outlier Count, and Inlier Ratio.
- Visualize filtered inliers vs rejected outliers.
- Export `inliers.csv` and `outliers.csv`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.magsac import MAGSACFilter

config = load_config()
df_matches = pd.read_csv("outputs/matches/matches.csv")
pts_src = df_matches[["src_x", "src_y"]].to_numpy(dtype=np.float32)
pts_ref = df_matches[["ref_x", "ref_y"]].to_numpy(dtype=np.float32)

magsac = MAGSACFilter(threshold_px=3.0, confidence=0.999, max_iters=10000)
filter_res = magsac.filter(pts_src, pts_ref)

print(f"Raw Matches: {len(pts_src)}")
print(f"Inlier Count: {filter_res['inlier_count']}")
print(f"Outlier Count: {filter_res['outlier_count']}")
print(f"Inlier Ratio: {filter_res['inlier_ratio']*100:.2f}%")
print(f"Estimated Homography Matrix:\n{filter_res['matrix']}")


In [ ]:
# Visualize Inliers vs. Outliers
plt.figure(figsize=(14, 6))

plt.scatter(filter_res['outliers_src'][:, 0], filter_res['outliers_src'][:, 1], c='red', marker='x', label=f'Outliers (N={filter_res["outlier_count"]})', alpha=0.6)
plt.scatter(filter_res['inliers_src'][:, 0], filter_res['inliers_src'][:, 1], c='green', marker='o', label=f'MAGSAC++ Inliers (N={filter_res["inlier_count"]})', alpha=0.8)

plt.title(f"MAGSAC++ Outlier Filtering (Inlier Ratio: {filter_res['inlier_ratio']*100:.1f}%)", fontsize=13, fontweight='bold')
plt.xlabel("Source X (px)")
plt.ylabel("Source Y (px)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/10_magsac_filtering.png", dpi=300)
plt.show()


In [ ]:
# Export inliers.csv and outliers.csv
df_inliers = pd.DataFrame(filter_res["inliers_src"], columns=["src_x", "src_y"])
df_inliers[["ref_x", "ref_y"]] = filter_res["inliers_ref"]
df_inliers.to_csv("outputs/matches/inliers.csv", index=False)

df_outliers = pd.DataFrame(filter_res["outliers_src"], columns=["src_x", "src_y"])
df_outliers[["ref_x", "ref_y"]] = filter_res["outliers_ref"]
df_outliers.to_csv("outputs/matches/outliers.csv", index=False)

print("Exported outputs/matches/inliers.csv and outliers.csv")
